In [1]:
#============================================================
# Celda 1 — Setup y rutas
#============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

RAW_DIR    = Path("output/conectividad/01-raw")
SILVER_DIR = Path("output/conectividad/02-silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Verificar que existen los parquets del 01
for f in ["raw_cobertura_2013_2020.parquet", "raw_cobertura_2021_2024.parquet"]:
    assert (RAW_DIR / f).exists(), f"❌ No encontrado: {RAW_DIR / f} — ejecuta primero el 01"

print("✅ Parquets RAW encontrados:")
for p in sorted(RAW_DIR.glob("*.parquet")):
    print(f"   {p.name}")

✅ Parquets RAW encontrados:
   raw_cobertura_2013_2020.parquet
   raw_cobertura_2021_2024.parquet
   raw_metadatos_conectividad.parquet


In [2]:
#============================================================
# Celda 2 — RAW 2013-2020 → formato largo
#============================================================
df_old = pd.read_parquet(RAW_DIR / "raw_cobertura_2013_2020.parquet")

ID_COLS   = ["Comunidad Autónoma", "Provincia"]
cols_100  = [c for c in df_old.columns if c not in ID_COLS]

df_old_long = df_old.melt(
    id_vars=ID_COLS,
    value_vars=cols_100,
    var_name="periodo",
    value_name="cob_100mbps"
)
df_old_long["año"] = df_old_long["periodo"].str.extract(r"(\d{4})").astype(int)

# Máximo por provincia×año (algunos años tienen jun + dic)
df_old_agg = (
    df_old_long
    .groupby(ID_COLS + ["año"], as_index=False)["cob_100mbps"]
    .max()
)

print(f"✅ 2013-2020 → shape: {df_old_agg.shape}")
print(f"   Años: {sorted(df_old_agg['año'].unique())}")
print(df_old_agg.head(3).to_string())

✅ 2013-2020 → shape: (416, 4)
   Años: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
  Comunidad Autónoma Provincia   año  cob_100mbps
0          ANDALUCÍA   Almería  2013     0.299160
1          ANDALUCÍA   Almería  2014     0.271518
2          ANDALUCÍA   Almería  2015     0.398375


In [3]:
#============================================================
# Celda 3 — RAW 2021-2024 → formato largo
#============================================================
df_new = pd.read_parquet(RAW_DIR / "raw_cobertura_2021_2024.parquet")

ID_COLS  = ["Comunidad Autónoma", "Provincia"]
cols_100 = [c for c in df_new.columns if c not in ID_COLS]

df_new_long = df_new.melt(
    id_vars=ID_COLS,
    value_vars=cols_100,
    var_name="periodo",
    value_name="cob_100mbps"
)
df_new_long["año"] = df_new_long["periodo"].str.extract(r"(\d{4})").astype(int)

df_new_agg = (
    df_new_long
    .groupby(ID_COLS + ["año"], as_index=False)["cob_100mbps"]
    .max()
)

print(f"✅ 2021-2024 → shape: {df_new_agg.shape}")
print(f"   Años: {sorted(df_new_agg['año'].unique())}")
print(df_new_agg.head(3).to_string())

✅ 2021-2024 → shape: (208, 4)
   Años: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Comunidad Autónoma Provincia   año  cob_100mbps
0          ANDALUCÍA   Almería  2021     0.857542
1          ANDALUCÍA   Almería  2022     0.882811
2          ANDALUCÍA   Almería  2023     0.925456


In [4]:
#============================================================
# Celda 4 — Concat 2013-2024, dedup, normalizar nombres
#============================================================
df = pd.concat([df_old_agg, df_new_agg], ignore_index=True)

# Drop duplicados: si un año aparece en ambos archivos, quedamos con el más reciente (2021-2024)
df = (
    df.sort_values(["Provincia", "año"])
    .drop_duplicates(subset=["Provincia", "año"], keep="last")
)

# Normalizar nombres (strip + title case)
df["Provincia"]          = df["Provincia"].str.strip().str.title()
df["Comunidad Autónoma"] = df["Comunidad Autónoma"].str.strip().str.title()

print(f"Shape final: {df.shape}")
print(f"Años cubiertos: {sorted(df['año'].unique())}")
print(f"Provincias: {df['Provincia'].nunique()}")
print(df.head(5).to_string())

Shape final: (624, 4)
Años cubiertos: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Provincias: 52
     Comunidad Autónoma Provincia   año  cob_100mbps
200  Castilla-La Mancha  Albacete  2013     0.525929
201  Castilla-La Mancha  Albacete  2014     0.529683
202  Castilla-La Mancha  Albacete  2015     0.537551
203  Castilla-La Mancha  Albacete  2016     0.536744
204  Castilla-La Mancha  Albacete  2017     0.621609


In [5]:
#============================================================
# Celda 5 — Guardar Silver parquet + CSV
#============================================================
df.to_parquet(SILVER_DIR / "cobertura_provincia_anio.parquet", index=False)
df.to_csv(SILVER_DIR    / "cobertura_provincia_anio.csv",     index=False)

print(f"✅ Guardado en {SILVER_DIR}:")
print(f"   cobertura_provincia_anio.parquet")
print(f"   cobertura_provincia_anio.csv")
print(f"\nShape: {df.shape}")
print(df.dtypes)

✅ Guardado en output/conectividad/02-silver:
   cobertura_provincia_anio.parquet
   cobertura_provincia_anio.csv

Shape: (624, 4)
Comunidad Autónoma        str
Provincia                 str
año                     int64
cob_100mbps           float64
dtype: object
